### **Earthquake_Event_Data-Gold_Layer**

In [0]:
%pip install reverse_geocoder

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.2 MB ? eta -:--:--
     ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.1/2.2 MB 1.7 MB/s eta 0:00:02
     ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/2.2 MB 4.6 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━ 1.1/2.2 MB 10.8 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 2.2/2.2 MB 18.0 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 15.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for reverse_geocoder: filename=reverse_geocoder-1.5.1-py3-none-any.whl size=2268066 sha256=455eafa54c2eb5ef89b264af3cf215ee18f75538463d8a8f736dd357451d2a05
  Stored in directory: /home/spark-621c45ff-1fd0-4e4a-bb00-0e/.cache/pip/wheels/17/3c/41/2bc89719586c2a5c53e9a527daa76a968a1288315c1ae2d904
Successfully built reverse_geocoder
Note: you may need to restart the kernel using %restart_python or dbutils.library.rest

In [0]:
spark.conf.set("fs.azure.account.auth.type.adlsweatherapidata.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.adlsweatherapidata.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.adlsweatherapidata.dfs.core.windows.net", "bf472d2f-5f2c-4d09-941f-7117dc63e911")
spark.conf.set("fs.azure.account.oauth2.client.secret.adlsweatherapidata.dfs.core.windows.net", "-xb8Q~WbkDR~RKt6JuL3qpKRclXrTDDrBcgurbsM")
spark.conf.set("fs.azure.account.oauth2.client.endpoint.adlsweatherapidata.dfs.core.windows.net", "https://login.microsoftonline.com/cc219978-6391-4153-8405-87dd0faa993b/oauth2/token")

In [0]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType
# Ensure the below library is installed on environment
import reverse_geocoder as rg

In [0]:
df = spark.read.table("earthquake_silver")

In [0]:
def get_country_code(lat, lon):

    coordinates = (float(lat), float(lon))
    return rg.search(coordinates) [0].get('cc')

In [0]:
# Registering the udf so they can be used on spark dataframe
get_country_code_udf = udf(get_country_code, StringType())

In [0]:
df_with_location = \
                   df.\
                       withColumn("country_code", get_country_code_udf(col("Latitude"), col("Longitude")))

In [0]:
# adding significance classification 
df_with_location_sig_class = \
                              df_with_location.\
                                  withColumn('sig_class',
                                             when(col("sig") < 100, "Low").\
                                             when((col("sig") >= 100) & (col("sig") < 500), "moderte").\
                                             otherwise("High")
                                            )

In [0]:
df_with_location_sig_class.write \
  .format("delta") \
  .mode("overwrite") \
  .save("abfss://dummydata@adlsweatherapidata.dfs.core.windows.net/weather/gold")

In [0]:
df_with_location_sig_class.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("earthquake_gold")

In [0]:
%sql
Select * from earthquake_gold

id,Longitude,Latitude,elevation,title,place,sig,mag,magType,time,updated,country_code,sig_class
ci11408890,-116.7776667,33.6633333,11.008,"M 1.3 - 10km SSW of Idyllwild, CA","10km SSW of Idyllwild, CA",26,1.29,ml,2014-01-01T23:51:36.02Z,2016-03-11T20:40:44.428Z,US,Low
ak01421ig3u,-151.6459,63.102,14.1,M 1.1 - Central Alaska,Central Alaska,19,1.1,ml,2014-01-01T23:47:26.501Z,2023-07-19T20:56:19.099Z,US,Low
ak01421i2zj,-150.0165,61.4581,44.6,"M 1.2 - 7 km SSW of Big Lake, Alaska","7 km SSW of Big Lake, Alaska",22,1.2,ml,2014-01-01T23:45:56.476Z,2023-07-19T20:56:17.82Z,US,Low
ak01421heui,-150.8276,62.7884,87.6,"M 1.4 - 32 km N of Petersville, Alaska","32 km N of Petersville, Alaska",30,1.4,ml,2014-01-01T23:42:43.623Z,2023-08-18T15:59:54.274Z,US,Low
usc000mnnn,-71.621,-29.888,40.0,"M 4.0 - 28 km WNW of Coquimbo, Chile","28 km WNW of Coquimbo, Chile",246,4.0,ml,2014-01-01T23:42:15Z,2014-03-07T00:25:54Z,CL,moderte
nc72134466,-122.7839966,38.8071671,0.013,"M 0.5 - 5 km WSW of Cobb, California","5 km WSW of Cobb, California",4,0.53,md,2014-01-01T23:40:44.02Z,2017-02-02T19:17:27.447Z,US,Low
nc72134461,-118.8948333,37.6433333,-0.446,"M 0.7 - 6 km E of Mammoth Lakes, California","6 km E of Mammoth Lakes, California",8,0.7,md,2014-01-01T23:38:03.87Z,2017-02-02T19:17:21.5Z,US,Low
ci11408882,-116.562,33.5175,8.148,"M 0.5 - 11km ESE of Anza, CA","11km ESE of Anza, CA",3,0.47,ml,2014-01-01T23:36:42.29Z,2016-03-11T09:29:49.278Z,US,Low
nn00557597,-118.4646,38.5678,8.7,"M 0.6 - 14 km ENE of Hawthorne, Nevada","14 km ENE of Hawthorne, Nevada",6,0.6,ml,2014-01-01T23:32:26.923Z,2018-06-29T22:59:45.339Z,US,Low
usb000m2w9,-176.8207,-23.9576,219.29,M 4.2 - south of the Fiji Islands,south of the Fiji Islands,271,4.2,mb,2014-01-01T23:32:17.53Z,2014-03-07T00:25:54Z,TO,moderte
